In [1]:
from DIC_LLM.agent_module import ask_agent_with_search
from dotenv import load_dotenv
load_dotenv() # hidden environment variables ignored by git
import logging
# Ignore Warnings of LLM-Outputs
class _NoNonTextWarning(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        message = record.getMessage()
        # If the log message contains our target string, drop it (return False)
        if "there are non-text parts in the response" in message:
            return False
        return True
logging.getLogger("google_genai.types").addFilter(_NoNonTextWarning())

In [55]:
# MAIN START 
#Overall Instruction
Team1 = "Bosnia Herzegovina" #EDIT HERE
Team2 = "Qatar" #EDIT HERE
intro = f"Within the next hours we have the fifa worldcup game between {Team1} vs {Team2}."

# 2. Define Bookmaker Baseline
bookmaker_odds = {
    "Source": "Consensus oddsportal.com",
    "Top_1": f"{Team1} 2-0 {Team2} (8)", #EDIT HERE
    "Top_2": f"{Team1} 1-0 {Team2} (9)", #EDIT HERE
    "Top_3": f"{Team1} 2-1 {Team1} (9)" #EDIT HERE
}


In [40]:
#Agent A
prompt_agent_a= intro + " "+f'''
You are a quantitative football data scientist analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
CRITICAL RULE (Zero Data Leakage): You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. When using your web search tool, restrict your queries to raw statistical databases (e.g., FBref, Transfermarkt, official FIFA stats). NEVER search for "predictions", "odds", or "betting tips". 
Chain-of-Thought Guardrail: Football is more or less a low-scoring game well approximated by independent Poisson distributions. 
Before outputting any probabilities, mathematically reason step-by-step about the base expected goals (lambda_Team1 and lambda_Team2) for this specific match. 
Variables & Weighting: Base your lambda estimation on these factors: Heavily weight recent match results (last 5-10 games) and short-term xG trends. 
Further factor in: overall attack/defensive strength, historical head-to-head, squad market value, Elo rating, days of rest, and travel distance. 
Neutral Ground Rule: Treat this match as being played on neutral ground with NO home advantage, unless Team 1 or Team 2 is a host nation (USA, Mexico, or Canada). 
Your Output should be strictly & only JSON: Return strictly as a JSON object with: "Poisson_Reasoning": Step-by-step mathematical logic calculating the base lambda_Team1 and lambda_Team2 based 
on the assigned quantitative metrics, explicitly noting recent form and neutral ground dynamics (max 5 sentences). 
"Top_1", "Top_2", "Top_3": The predicted exact scores and their probabilities.
'''

agent_a_response = ask_agent_with_search(prompt_agent_a)
print(agent_a_response)



```json
{
  "Poisson_Reasoning": "I estimated the base expected goals (lambda) by evaluating recent tournament form, overall squad strength, and the neutral ground setting in Seattle. Bosnia and Herzegovina's lambda is set higher due to Qatar's recent defensive struggles and key suspensions. Qatar's lambda reflects their lower attacking output and historical Elo rating disadvantage. These lambdas were then fed into independent Poisson distributions to calculate the exact score probabilities.",
  "Top_1": "1-0 (Bosnia Herzegovina wins) - 13.3%",
  "Top_2": "2-0 (Bosnia Herzegovina wins) - 12.0%",
  "Top_3": "1-1 (Draw) - 10.7%"
}
```


In [41]:
#formatting corrections only (Agent A)
{
    "Poisonon Reasoning": "Portugal\'s elite squad market value and high Elo rating, combined with a dominant recent xG trend of 2.4 per game, establish a strong baseline attack. Uzbekistan\'s lower Elo and recent historical data against top-tier UEFA opposition yield a heavily discounted offensive baseline, despite respectable AFC defensive metrics. Adjusting for the neutral ground in North America, equal travel distance, and Portugal\'s robust defensive xGA of 0.7, Uzbekistan\'s expected goals are severely suppressed. Applying these weighted factors, the calculated base expected goals are lambda_POR = 2.30 and lambda_UZB = 0.40. Utilizing independent Poisson distributions with these lambdas generates the exact score probabilities, heavily favoring a clean sheet victory for Portugal..",
    "Top_1": {
    "Score": "2-0",
    "Probability": "18.5%"
    },
    "Top_2": {
    "Score": "1-0",
    "Probability": "14.2%"
    },
    "Top_3": {
    "Score": "3-0",
    "Probability": "11.8%"
    }
}

{'Poisonon Reasoning': "Portugal's elite squad market value and high Elo rating, combined with a dominant recent xG trend of 2.4 per game, establish a strong baseline attack. Uzbekistan's lower Elo and recent historical data against top-tier UEFA opposition yield a heavily discounted offensive baseline, despite respectable AFC defensive metrics. Adjusting for the neutral ground in North America, equal travel distance, and Portugal's robust defensive xGA of 0.7, Uzbekistan's expected goals are severely suppressed. Applying these weighted factors, the calculated base expected goals are lambda_POR = 2.30 and lambda_UZB = 0.40. Utilizing independent Poisson distributions with these lambdas generates the exact score probabilities, heavily favoring a clean sheet victory for Portugal..",
 'Top_1': {'Score': '2-0', 'Probability': '18.5%'},
 'Top_2': {'Score': '1-0', 'Probability': '14.2%'},
 'Top_3': {'Score': '3-0', 'Probability': '11.8%'}}

In [44]:
#Agent B
prompt_agent_b= intro + " "+f'''
You are an news-based investigative football scout analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
CRITICAL RULE (Zero Data Leakage):Use web search for the latest news (last 48h) on the teams. You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. Use web search specifically to find the latest news (last 48h), press conferences, and injury reports. You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. Actively filter out any market sentiment from the articles you read.
Chain-of-Thought Guardrail: Football is more or less a low-scoring game well approximated by independent Poisson distributions. Before outputting any probabilities, mathematically reason step-by-step about how current intelligence shifts 
the expected goals (lambda_Team1 and lambda_Team2) from a statistical baseline. 
Variables: Base your lambda shifts strictly on qualitative conditions: player availability/injuries, tactical matchups, pressing intensity, set-piece efficiency, climate adaptation (heat/humidity/altitude), squad depth, and big-match experience. 
Include "Host Nation Crowd Psychology" only if USA, Mexico, or Canada is playing. 
Your Output should be strictly & only JSON: "Narrative_to_Poisson_Mapping": Step-by-step reasoning on how the retrieved intelligence specifically alters lambda_Team1 and lambda_Team2 without inflating overall goal expectations (max 5 sentences). 
"Top_1", "Top_2", "Top_3": The predicted exact scores and their probabilities.
'''
agent_b_response = ask_agent_with_search(prompt_agent_b)
print(agent_b_response)

{
  "Narrative_to_Poisson_Mapping": "Bosnia and Herzegovina's expected goals (lambda) see a slight upward shift due to their physical and aerial advantages against a depleted Qatar side. Qatar's lambda decreases significantly because they are missing two suspended players following red cards in their recent 6-0 defeat to Canada, severely weakening their squad depth and tactical setup. Both teams are desperate for a win to advance, but Qatar's defensive vulnerabilities and lack of discipline make them highly susceptible to Bosnia's structured attacks. Consequently, the baseline Poisson distributions are adjusted to favor a low-scoring but decisive Bosnian victory.",
  "Top_1": {
    "Score": "2-0",
    "Probability": "14.5%"
  },
  "Top_2": {
    "Score": "1-0",
    "Probability": "12.1%"
  },
  "Top_3": {
    "Score": "2-1",
    "Probability": "9.4%"
  }
}


In [ ]:
#formatting corrections only (Agent B)
agent_b_response = '''
{
    "Narrative_to_Poisson_Mapping": "Portugal's unexpected draw against DR Congo increases their attacking urgency, leveraging their superior squad depth and big-match experience to slightly elevate their expected goals. Uzbekistan demonstrated attacking capability in their opener but remains defensively vulnerable against elite tactical setups. The climate conditions in Houston and the tactical mismatch favor Portugal's possession control, suppressing Uzbekistan's scoring baseline. Overall goal expectations remain conservative, reflecting Portugal's recent trend of lower-scoring matches.",
    "Top_1": {
    "Score": "2-0",
    "Probability": "18.5%"
    },
    "Top_2": {
    "Score": "1-0",
    "Probability": "14.2%"
    },
    "Top_3": {
    "Score": "3-0",
    "Probability": "11.8%"
    }
}
'''

In [ ]:
#Agent C, no probabilities necessary because no structured probabilities are required, raw text containing thoughts is sufficient. 

prompt_agent_c= intro + " "+f'''
You are the Skeptic. Evaluate the independent analyses: Quant [Input A: {agent_a_response}] and Qual [Input B: {agent_b_response}]. 
CRITICAL RULE (Zero Data Leakage): Under no circumstances use betting markets or consensus favorites to validate or invalidate A and B. When using your web search, use it STRICTLY to fact-check the specific claims made in Input A and B (e.g., verifying if a player is actually injured, or if an xG stat is accurate). Do not search for general match predictions. 
Chain-of-Thought Guardrail: You try to find potential weaknesses of Input A (Agent A) and Input B (Agent B) provided. 
Your Output: Return strictly as a JSON object with: "Critique_Reasoning": 
Step-by-step logical deconstruction of the lambda assumptions in both models (max 5 sentences). 
"Critique_Quant": A concise, targeted critique of Agent A's variables (focusing on recent form vs. historical data). "Critique_Qual": 
A concise, targeted critique of Agent B's variables.
'''
agent_c_response = ask_agent_with_search(prompt_agent_c)
print(agent_c_response)

In [50]:
agent_c_response = '''
{
  "Critique_Reasoning": "The lambda assumptions in both models diverge significantly on how to weight baseline historical metrics versus acute situational factors. Agent A uses a traditional Poisson approach grounded in historical Elo and general form, which risks ignoring the immediate structural deficit caused by Qatar's two recent red cards. Agent B aggressively shifts the expected goals based on the narrative of Qatar's 6-0 loss to Canada and their depleted squad, assuming Bosnia can easily exploit these weaknesses. However, Agent B's assumption of Bosnia's attacking competence is not well-supported by Bosnia's own recent struggles, including a heavy 4-1 loss to Switzerland. Ultimately, Agent A under-reacts to critical recent roster data, while Agent B over-reacts to the narrative of Qatar's collapse without verifying Bosnia's actual offensive capability.",
  "Critique_Quant": "Agent A relies too heavily on historical Elo ratings and base expected goals, failing to sufficiently weight the drastic recent form changes—specifically Qatar's 6-0 collapse to Canada and the immediate mathematical impact of losing two key players (Assim Madibo and Homam Ahmed) to suspension.",
  "Critique_Qual": "Agent B overly relies on the narrative of Qatar's missing players to artificially inflate Bosnia's expected goals, ignoring that Bosnia's recent form lacks empirical evidence to support the 'structured attacks' and offensive dominance assumed in the qualitative adjustment."
}
'''

In [49]:
print(prompt_agent_c)

Within the next hours we have the fifa worldcup game between Bosnia Herzegovina vs Qatar. 
You are the Skeptic. Evaluate the independent analyses: Quant [Input A: ```json
{
  "Poisson_Reasoning": "I estimated the base expected goals (lambda) by evaluating recent tournament form, overall squad strength, and the neutral ground setting in Seattle. Bosnia and Herzegovina's lambda is set higher due to Qatar's recent defensive struggles and key suspensions. Qatar's lambda reflects their lower attacking output and historical Elo rating disadvantage. These lambdas were then fed into independent Poisson distributions to calculate the exact score probabilities.",
  "Top_1": "1-0 (Bosnia Herzegovina wins) - 13.3%",
  "Top_2": "2-0 (Bosnia Herzegovina wins) - 12.0%",
  "Top_3": "1-1 (Draw) - 10.7%"
}
```] and Qual [Input B: {
  "Narrative_to_Poisson_Mapping": "Bosnia and Herzegovina's expected goals (lambda) see a slight upward shift due to their physical and aerial advantages against a depleted Q

In [ ]:
# Agent E
prompt_agent_e= intro + " "+f'''
You are the Meta-Analyst and Final Adjudicator analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
Synthesize the independent analyses from the Quant Agent [Input A: {agent_a_response}], the Qual Agent [Input B: {agent_b_response}], and the Skeptic [Input C: {agent_c_response}] 
for the FIFA World Cup match. 
CRITICAL RULE (Zero Data Leakage & Search Directive): Under no circumstances use betting markets, odds, or consensus favorites to calibrate your final prediction. 
If you use your web search tool, use it STRICTLY to resolve factual deadlocks (e.g., checking if a player is actually injured after conflicting reports).
NEVER search for "match predictions", "betting odds", or "consensus picks". Actively filter out any market sentiment from search results. 
Anti-Averaging & True Adjudication Guardrail: Do NOT simply average the lambdas or probabilities of Agents A and B. You must critically evaluate ALL three inputs.
REMEMBER: Agent C (The Skeptic) is not infallible and can also be wrong. 
If Agent C's critique is logically flawed, overly dismissive, or ignores strong data, you must boldly overrule Agent C and defend the valid insights from A or B. 
If Agent C correctly identifies flaws, adjust accordingly. If all three agents (A, B, and C) are flawed, independently formulate a new, accurate lambda recalibration. 
Chain-of-Thought Guardrail: Use simulated Bayesian updating to form a final probability distribution. 
Your Output should be strictly & only JSON: Return strictly as a JSON object with: "Bayesian_CoT": Step-by-step logical synthesis explaining how the final lambdas were calibrated, explicitly stating whose logic (including C's) you validated, overruled, or if you had to independently recalibrate (max 5 sentences). "Top_1", "Top_2", "Top_3": The final calibrated exact scores and their probabilities.
'''
agent_e_response = ask_agent_with_search(prompt_agent_e)
#print(agent_e_response)

In [52]:
print(prompt_agent_e)

Within the next hours we have the fifa worldcup game between Bosnia Herzegovina vs Qatar. 
You are the Meta-Analyst and Final Adjudicator analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
Synthesize the independent analyses from the Quant Agent [Input A: ```json
{
  "Poisson_Reasoning": "I estimated the base expected goals (lambda) by evaluating recent tournament form, overall squad strength, and the neutral ground setting in Seattle. Bosnia and Herzegovina's lambda is set higher due to Qatar's recent defensive struggles and key suspensions. Qatar's lambda reflects their lower attacking output and historical Elo rating disadvantage. These lambdas were then fed into independent Poisson distributions to calculate the exact score probabilities.",
  "Top_1": "1-0 (Bosnia Herzegovina wins) - 13.3%",
  "Top_2": "2-0 (Bosnia Herzegovina wins) - 12.0%",
  "Top_3": "1-1 (Draw) - 10.7%"
}
```], the Qual Agent [Input B: {
  "Narrative_to_Poisson_Mapping": "Bosnia

In [65]:
#Formatting corrections only (Agent E)
agent_e_response = '''
{
  "Bayesian_CoT": "We validate Agent C's critique that Agent A under-reacts to Qatar's critical suspensions (Madibo and Ahmed), while Agent B over-inflates Bosnia's attacking efficiency despite their recent 4-1 loss to Switzerland. By updating the joint probability distribution, we independent-recalibrate the expected goal parameters (lambda) to reflect Qatar's structural defensive regression without overestimating Bosnia's offensive capabilities. This intermediate calibration accounts for a tight, lower-tempo fixture where Qatar deploys a low block to mitigate their missing depth. Consequently, the posterior probabilities shift toward a highly controlled, low-scoring Bosnian victory or a low-scoring stalemate.",
  "Top_1": "1-0 (Bosnia Herzegovina wins) - 14.1%",
  "Top_2": "2-0 (Bosnia Herzegovina wins) - 11.8%",
  "Top_3": "1-1 (Draw) - 10.5%"
}
'''

In [67]:
#Central Ledger
import json
from datetime import datetime, timezone

def safe_parse_json(agent_name: str, raw_string: str) -> dict:
    """Helper method to safely parse LLM output into a dictionary."""
    try:
        # Strip potential markdown formatting that LLMs sometimes add (e.g., ```json ... ```)
        clean_string = raw_string.strip()
        if clean_string.startswith("```json"):
            clean_string = clean_string[7:-3].strip()
            
        return json.loads(clean_string)
    except json.JSONDecodeError:
        print(f"Warning: Failed to parse JSON for {agent_name}. Saving as raw text instead.")
        return {"Only Raw_Text Available": raw_string}


def log_full_match_to_ledger(match_id: str, outputs: dict, bookmaker_data: dict, ledger_path: str = "predictions_ledger.jsonl"):
    """
    Saves the complete state of the DAG (All Agents + Bookmaker) into a single ledger entry.
    """
    
    # 1. Safely parse all agent outputs
    parsed_agents = {
        "Agent_A_Quant": safe_parse_json("Agent_A", outputs.get('agent_a', '{}')),
        "Agent_B_Qual": safe_parse_json("Agent_B", outputs.get('agent_b', '{}')),
        "Agent_C_Critique": safe_parse_json("Agent_C", outputs.get('agent_c', '{}')),
        "Agent_E_Meta": safe_parse_json("Agent_E", outputs.get('agent_e', '{}'))
    }

    # 2. Build the master record
    master_record = {
        "metadata": {
            "match_id": match_id,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        },
        "bookmaker_baseline": bookmaker_data,
        "dag_pipeline": parsed_agents
    }
    
    # 3. Append to the JSONL file
    with open(ledger_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(master_record) + '\n')
        
    print(f"Successfully logged full DAG prediction for {match_id} to {ledger_path}")

# ==========================================
# Example Usage in your Orchestrator (main.py):
# ==========================================

# 1. Run your agents (Simulated strings here)
out_a = agent_a_response
out_b = agent_b_response
out_c = agent_c_response
out_e = agent_e_response



# 2. Pack agent outputs into a dictionary
agent_results = {
    "agent_a": out_a,
    "agent_b": out_b,
    "agent_c": out_c,
    "agent_e": out_e
}
    
# 3. Log everything at once!
log_full_match_to_ledger(
    match_id=f"{Team1}-{Team2}", 
    outputs=agent_results, 
    bookmaker_data=bookmaker_odds
)

Successfully logged full DAG prediction for Bosnia Herzegovina-Qatar to predictions_ledger.jsonl
